# 05 Pretrained ResNet18, fine-tuned: RGB versus 13 bands

**Purpose.** Fine-tune a ResNet18 pretrained on ImageNet, once on RGB (3 channels) and once on all 13 bands, with the same split, preprocessing, augmentation and seeds as notebooks 03 and 04. This tests whether the 13-band advantage still holds when the network does not start from zero but from features learned on 1.2 million photos (SQ1). The two inputs differ only in the first layer of the network.

**Functions.** `src/piece_of_crop/resnet.py` (`build_resnet`, `resnet_param_groups`, `IMAGE_SIZE`), `train.py` (`seed_all`, `train_model`, `predict`, `predictions_table`, the same as for the CNN) and `data.py` (`make_loaders`).

**Reads.** `results/split.csv`, `results/band_stats.csv` and the patch cache `data/eurosat_cache.npy`. The ImageNet weights are downloaded by torchvision on first use.

**Saves.** `results/resnet_results.csv`, `results/resnet_history.csv`, six test prediction files `results/predictions/resnet_<input>_seed<n>.csv`, the training curves `results/figures/resnet_<input>_training.png` and the trained weights in `models/` (not in Git; notebook 07 uses them).

## 1. Setup and shared data from notebook 02

All functions come from the installed `piece_of_crop` package. On a Mac the network trains on the GPU through MPS if PyTorch supports it, otherwise on the CPU.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, f1_score
from IPython.display import display

from piece_of_crop import CLASS_NAMES, RGB_BAND_INDICES, read_split, get_cache, make_loaders
from piece_of_crop import IMAGE_SIZE, build_resnet, resnet_param_groups
from piece_of_crop import seed_all, train_model, predict, predictions_table

# hyperparameters
SEEDS = (0, 1, 2)
BATCH_SIZE = 32
EPOCHS = 15  # fixed, the cosine schedule runs over all of them
LR_BACKBONE = 1e-4  # pretrained layers
LR_HEAD = 1e-3  # new 10-class head
WEIGHT_DECAY = 1e-4

project_root = Path.cwd().parents[0]
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in [results_dir, figures_dir, predictions_dir, models_dir]:
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
label_ids = np.arange(len(CLASS_NAMES))
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

print("Project root:", project_root)
print("Device:", device)
print("Patches:", split["split"].value_counts().to_dict())
print("Image size:", IMAGE_SIZE)

## 2. The network

ResNet18 is a convolutional network of 18 layers built from residual blocks. Each block adds its input to its output through a "skipping connection", which "makes gradient flow easier" in a deep network (lecture_6ses.8_intro_to_DL-2.pdf, slide 38). Its convolution and pooling layers are the feature extractor and the last fully connected layer is the classifier (lecture_12_finetuning and transfer learning.pdf, slide 4). Trained on ImageNet, the feature extractor has learned edges, textures and parts of objects from 1.2 million photos (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, section 0). We keep it and replace the 1000-class head with a new linear layer with 10 outputs, as in the course (lecture_12_finetuning and transfer learning.pdf, slide 8).

**Claim.** For 13 bands, the first layer gets the pretrained red, green and blue filters at the positions of B04, B03 and B02, and filters of zero for the other 10 bands. This makes the 13-band network start exactly as the RGB network.

**Reasons.** The pretrained first layer only has filters for 3 colour channels, so the 10 other bands need new filters. We decided between three ways to start them: at zero, at random, or as copies of the average colour filter. With zeros, the 10 extra bands add nothing at the start, so the 13-band network gives exactly the same output as the RGB network. Any difference after training comes from what the network learns from the extra bands, which is the question of the project.

**Evidence.** The cell below builds both networks from the ImageNet weights and gives them the same patch: the outputs are identical. The course does not cover this step, so we rely on external work. When a pretrained model was extended to multispectral Sentinel-2 input, starting the new channels at zero worked better than the average of the colour filters in all six tasks tested (external: Marimo et al., 2025, "Beyond the Visible: Multispectral Vision-Language Learning for Earth Observation", arXiv:2503.15969, section 6.2). Random filters for the missing channels gave no systematic improvement and made fine-tuning take longer (external: Lacoste et al., 2023, "GEO-Bench: Toward Foundation Models for Earth Monitoring", arXiv:2306.03831, section 6.2.4).

**Warrant.** A comparison only shows the effect of one change if everything else is the same (our rule of changing one thing at a time). Starting both inputs from the same network is the closest we can get to that with a pretrained model.

**Counterargument.** Zero filters must be learned from nothing with the small backbone learning rate, while copies of the colour filters would give the extra bands a head start (external: Corley et al., 2023, arXiv:2305.13456, repeat the colour filters for all bands). The external evidence for zeros also comes from a different kind of model (a vision-language model, not a ResNet). We accept both points. Section 6 shows whether the 13-band network has learned to use the extra bands within the 15 epochs.

In [ ]:
# check both networks: 10 outputs, and the 13-band network starts as the RGB network
rgb_check = build_resnet(in_channels=3).eval()
ms_check = build_resnet(in_channels=13).eval()
ms_check.fc.load_state_dict(rgb_check.fc.state_dict())  # same head, so conv1 is the only difference

x = torch.randn(2, 13, IMAGE_SIZE, IMAGE_SIZE)
with torch.no_grad():
    out_rgb = rgb_check(x[:, RGB_BAND_INDICES])  # B04, B03, B02
    out_ms = ms_check(x)
assert out_ms.shape == (2, 10)
print("Largest difference between the RGB and 13-band outputs:", (out_rgb - out_ms).abs().max().item())

for name, check in (("rgb", rgb_check), ("13band", ms_check)):
    n_params = sum(p.numel() for p in check.parameters() if p.requires_grad)
    print(f"{name}: {n_params:,} trainable parameters")
del rgb_check, ms_check

## 3. Training settings

| Setting | Value | Why |
|---|---|---|
| What is trained | every layer (full fine-tuning) | see below |
| Loss | cross-entropy on the logits | as in notebook 04 (lecture_5_intro_to_DL.pdf, slide 25) |
| Optimizer | AdamW, learning rate 0.0001 for the pretrained layers and 0.001 for the new head | the backbone gets "a 10× smaller LR than the new head, so we don't destroy what it learned" (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, section 4, strategy C); "use lower learning rates than training from scratch" (lecture_12_finetuning and transfer learning.pdf, slide 6) |
| Weight decay | 0.0001 | "use weight decay to maintain pre-trained knowledge" (slide 6); the same value as the course notebook |
| Learning-rate schedule | cosine over all batches | "consider learning rate scheduling" (slide 6); the course notebook uses `CosineAnnealingLR` |
| Epochs | 15, fixed | see below |
| Best epoch | lowest validation loss, kept at the end | the test set is used only after this choice (lecture_2_introduction_to_ML.pdf, slide 24) |
| Batch size | 32 | see below |
| Image size | 224 x 224 | see below |
| Normalisation | our per-band training statistics | see below |
| Augmentation | random horizontal and vertical flips, training only | the same as notebook 04 (see notebook 02, section 6) |

**Fine-tuning strategy.** We decided between the three strategies of the course: training from scratch, training only a new head on a frozen backbone ("feature extraction"), and full fine-tuning, where every layer is trained (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, section 4), and the partial unfreezing of the last block from its exercises (section 6). *Claim:* full fine-tuning fits our data best. *Reasons:* the course's guide is that little data calls for feature extraction, while "lots of data" from a "different domain" calls for "full fine-tuning or train from scratch" (same notebook, quiz table). We have about 1,900 training patches per class, inside the 1 to 5 thousand images per class with which a model can be trained from scratch (lecture_12_finetuning and transfer learning.pdf, slide 3), and satellite patches look very different from ImageNet photos. *Evidence:* our CNN trained from scratch already reaches 91.8% (RGB) and 95.4% (13 bands) on test (notebook 04), so we have enough data to train every layer. In the course experiment, full fine-tuning beat feature extraction (90.7% against 79.5% test accuracy) even with only 10 images per class. *Warrant:* features learned on photos must be adapted to satellite images, and only layers that are trained can adapt (slide 5: "let some (or all) of the feature extracting layers be retrained"). *Counterargument:* full fine-tuning needs more compute and has a "higher risk of forgetting/overfitting" (same notebook, section 4). We accept the compute cost and limit the risk with the small backbone learning rate, weight decay and the best validation epoch.

**Epochs: 15 or 30.** We decided between the 30 epochs of notebook 04 and the 15 of the course's fine-tuning notebook. A pretrained network starts from useful features, so it needs fewer epochs than a network trained from scratch: in the course run, the validation loss of full fine-tuning hardly changed after epoch 10 (same notebook, strategy C). Section 5 shows whether this holds for our data.

**Batch size: 32 or 128.** We decided between 128 (notebook 04) and 32 (the course's fine-tuning notebook). At 224 x 224 a ResNet18 holds far more activations per patch in memory than the small CNN at 64 x 64, so a batch of 128 is a risk on a laptop GPU. With 32 patches, a batch is under 0.2% of the training set, so the gradient estimate stays noisy (Mathematics for Machine Learning, section 7.1, page 232). RGB and 13 bands use the same batch size, so the comparison between them stays fair.

**Image size: 64, 128 or 224.** *Claim:* we resize the 64 x 64 patches to 224 x 224. *Reasons:* the course preprocesses new images "the same way as the original training" of the pretrained model, which is 224 x 224 for ImageNet (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, section 1; lecture_12_finetuning and transfer learning.pdf, slide 6, point 1). ResNet18 shrinks its input 32 times, so a 64 x 64 patch would end as a 2 x 2 feature map, while a 224 x 224 patch ends as 7 x 7, the size its last layers were trained on. *Evidence:* on EuroSAT, resizing from 64 to 224 raised the accuracy of frozen ImageNet features with a nearest-neighbour classifier from 82% to 91% for RGB and from 79% to 90% for all bands (external: Corley et al., 2023, "Revisiting pre-trained remote sensing model benchmarks: resizing and normalization matters", arXiv:2305.13456, table 1). *Counterargument:* resizing adds no new information and makes every patch 12 times larger, so training is much slower. These numbers come from frozen features, not fine-tuning, so the gain for a fine-tuned network may be smaller. We accept the cost because the pretrained features work best at the scale they were learned at.

**Normalisation: ImageNet or our own statistics.** *Claim:* we keep the per-band standardisation of notebook 02 instead of the ImageNet mean and standard deviation. *Reasons:* the purpose of the course rule is that the pretrained filters receive inputs on the scale they were trained on. The ImageNet values (for example a mean of 0.485 for red) assume photo brightness between 0 and 1, and bring each colour channel to roughly mean 0 and spread 1. Our raw values are reflectances of about 0 to 10,000, so they would first need an arbitrary rescaling. Standardising each band with our training statistics reaches the same scale directly, and ImageNet has no statistics for 10 of our 13 bands. *Evidence:* on EuroSAT, channel-wise standardisation together with resizing to 224 raised the RGB accuracy of ImageNet features from 66% to 91% compared with dividing by 10,000 (external: Corley et al., 2023, introduction). *Warrant:* a network can only reuse its filters if the inputs are on the scale it learned with. *Counterargument:* the course asks to normalise "with the ImageNet mean/std and not with the Flowers mean/std" (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, quiz question 1), and slide 6 asks for the same preprocessing as the original data. We accept that we deviate from the course method, but not from its goal: the deviation is needed because our data are not photos.

## 4. Train RGB and 13-band networks with three seeds

For every seed the data loaders are created again, so the shuffling and augmentation follow that seed. After training, each network is evaluated on validation and test with the weights of its best epoch. The training accuracy is the one measured in the training loop at that epoch. The time of each run is recorded, as in the course notebook. The test predictions use the team format from `predictions_table`, the same as notebooks 03 and 04.

**Before running all six networks:** watch how long the first epoch of the first network takes. If six runs of 15 epochs would take too long, stop the cell and tell Claude, so we can decide on a smaller image size.

In [ ]:
history_runs = []
result_rows = []
test_files = split.loc[split["split"].eq("test"), "file"]
test_labels = split.loc[split["split"].eq("test"), "label"].to_numpy()

# train every model, 2 inputs x 3 seeds
for input_type, n_channels in (("rgb", 3), ("13band", 13)):
    for seed in SEEDS:
        print(f"\n=== ResNet18 {input_type}, seed {seed} ===")
        seed_all(seed)
        loaders = make_loaders(split, cache, band_stats, input_type, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE)

        model = build_resnet(in_channels=n_channels, num_classes=len(CLASS_NAMES)).to(device)
        optimizer = torch.optim.AdamW(resnet_param_groups(model, LR_BACKBONE, LR_HEAD), weight_decay=WEIGHT_DECAY)
        # the model ends with the weights of the epoch with the lowest val loss
        start = time.time()
        history, best_epoch = train_model(model, loaders["train"], loaders["val"], optimizer, EPOCHS, device)
        minutes = (time.time() - start) / 60
        print(f"Training took {minutes:.1f} minutes")

        run = pd.DataFrame(history)
        run["epoch"] = np.arange(1, EPOCHS + 1)
        run["input"], run["seed"] = input_type, seed
        history_runs.append(run)
        torch.save(model.state_dict(), models_dir / f"resnet_{input_type}_seed{seed}.pt")

        # train accuracy comes from the training loop at the best epoch
        result_rows.append({"model": "ResNet18", "input": input_type, "seed": seed, "split": "train",
            "accuracy": history["train_acc"][best_epoch - 1], "best_epoch": best_epoch, "minutes": minutes})
        # best epoch is picked on validation, test set is only used from here
        for subset in ("val", "test"):
            y_true, proba = predict(model, loaders[subset], device)
            y_pred = proba.argmax(axis=1)
            result_rows.append({"model": "ResNet18", "input": input_type, "seed": seed, "split": subset,
                "accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro"),
                "best_epoch": best_epoch, "minutes": minutes})
            # save the test predictions to csv
            if subset == "test":
                assert np.array_equal(y_true, test_labels)  # predictions are in split order
                predictions_table(test_files, y_true, proba).to_csv(
                    predictions_dir / f"resnet_{input_type}_seed{seed}.csv", index=False)


resnet_history = pd.concat(history_runs, ignore_index=True)
resnet_results = pd.DataFrame(result_rows)

resnet_history.to_csv(results_dir / "resnet_history.csv", index=False)
resnet_results.to_csv(results_dir / "resnet_results.csv", index=False)

## 5. Training curves

Training and validation loss per epoch for each seed (left) and validation accuracy (right). The training loss is measured on augmented patches while the network is still changing during the epoch, so it is not directly comparable with the validation loss, and the training accuracy in section 6 is a rough check for overfitting, not an exact score.

In [ ]:
# plot the loss and accuracy for every epoch
for input_type in ("rgb", "13band"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for seed in SEEDS:
        run = resnet_history[(resnet_history["input"] == input_type) & (resnet_history["seed"] == seed)]
        axes[0].plot(run["epoch"], run["train_loss"], ls="--", alpha=0.6, label=f"train seed {seed}")
        axes[0].plot(run["epoch"], run["val_loss"], label=f"val seed {seed}")
        axes[1].plot(run["epoch"], run["val_acc"], label=f"seed {seed}")

    axes[0].set(title=f"ResNet18 {input_type}: loss per epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[1].set(title=f"ResNet18 {input_type}: validation accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figures_dir / f"resnet_{input_type}_training.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Best epoch (lowest validation loss) and training time for each network:")
display(resnet_results[resnet_results["split"].eq("test")][["input", "seed", "best_epoch", "minutes"]].round(1))

## 6. Results: RGB versus 13 bands

Mean and standard deviation over the three seeds, for training, validation and test. The per-class reports show seed 0 only; the other seeds are in the prediction files.

In [ ]:
# mean and std over the 3 seeds
summary = resnet_results.groupby(["input", "split"], sort=False)[["accuracy", "macro_f1"]]
display(summary.agg(["mean", "std"]).round(4))

# per class report, only for seed 0
for input_type in ("rgb", "13band"):
    pred = pd.read_csv(predictions_dir / f"resnet_{input_type}_seed0.csv")
    print(f"\nPer class test report: {input_type}, seed 0")
    print(classification_report(pred["y_true"], pred["y_pred"], labels=label_ids,
          target_names=list(CLASS_NAMES), zero_division=0))

## 7. Limitations of the pretrained model

- **ImageNet favours RGB.** Only the 3 colour channels of the first layer are pretrained; the filters for the other 10 bands are learned from our training patches alone. The course advises choosing "a pre-trained model trained on related domain" (lecture_12_finetuning and transfer learning.pdf, slide 6), and ImageNet photos are not satellite images. So this notebook tests whether 13 bands help a network pretrained on RGB photos, not a network pretrained on multispectral data. Networks pretrained on Sentinel-2 images exist and gave a modest gain over ImageNet weights (external: Lacoste et al., 2023, arXiv:2306.03831, section 6.2.4); using one is a next step.
- **Resizing adds no information.** The 224 x 224 patches are upsampled 64 x 64 patches, so the network sees the same 4,096 pixels, only at the scale its filters expect.
- **The data are land-cover labels, not claims.** The models predict what a patch shows, not crop losses or yields, so they support human review of a declaration and do not decide it.
- **The spatial split reduces but does not remove leakage.** Validation and test are different regions (notebook 02), and three seeds show the variation from training, not a full statistical test; notebook 06 adds that test.